In [ ]:
import ipywidgets as wg
from ipywidgets import HBox, VBox, Text
from IPython.display import display
import os
import matplotlib
import xlrd
import pandas as pd
import csv
import time
import numpy as np
from scipy import interpolate
from numpy import trapz
import matplotlib.pyplot as plt
from functools import reduce
%matplotlib inline
import matplotlib.lines as mlines

half_cell_dir = os.getcwd() + '/CDT Demo/Half Cell Voltage Curves'
cathode_options = ['Faradion_Gen2_4.25V', 'Faradion_Gen2_4.35V', 'Faradion_Gen2_4.1V', 'Suxiang_XN33S', 'Standard Potential_SodaCAM-001', 'NFPP_2', 'Prussian White', 'NVP', 'NVPF', 'LFP', 'BTR_M2-C', 'T61A']
anode_options = ['Faradion_HC', 'Faradion_HC_commercial', 'LongTime_SHC300', 'Kaijin_AS-C12','MAGE-3', 'Kuraray_Kuranode (Type I)', 'Tin Powder_DGME', 'ShowaDenko_MAGE-3', 'Anode Free', 'Pb Powder']

def savitzky_golay(y, window_size, order, deriv=0, rate=1):
    import numpy as np
    from math import factorial

    try:
        window_size = np.abs(np.int(window_size))
        order = np.abs(np.int(order))
    except (ValueError, msg):
        raise ValueError("window_size and order have to be of type int")
    if window_size % 2 != 1 or window_size < 1:
        raise TypeError("window_size size must be a positive odd number")
    if window_size < order + 2:
        raise TypeError("window_size is too small for the polynomials order")
    order_range = range(order+1)
    half_window = (window_size -1) // 2
    b = np.mat([[k**i for i in order_range] for k in range(-half_window, half_window+1)])
    m = np.linalg.pinv(b).A[deriv] * rate**deriv * factorial(deriv)
    firstvals = y[0] - np.abs( y[1:half_window+1][::-1] - y[0] )
    lastvals = y[-1] + np.abs(y[-half_window-1:-1][::-1] - y[-1])
    y = np.concatenate((firstvals, y, lastvals))
    return np.convolve( m[::-1], y, mode='valid')

align_kw = dict(
    _css = (('.widget-label', 'min-width', '20ex'),),
    margin = '0px 0px 5px 12px'
)


#Cell Level Details
u_max = wg.FloatText(value=4.2, layout=wg.Layout(width='40%'))   #### dash level
u_min = wg.FloatText(value=1.0, layout=wg.Layout(width='40%'))   #### dash level 
n_p_ratio = wg.FloatText(value=1.13, layout=wg.Layout(width='40%'))   #### in cell class
max_voltage_box = wg.HBox([wg.Label(value='Balance Voltage:'), u_max, wg.Label(value='V')])   #### dash level
min_voltage_box = wg.HBox([wg.Label(value='Min Voltage:'), u_min, wg.Label(value='V')])       #### dash level
n_p_ratio_box = wg.HBox([wg.Label(value='N/P Ratio'), n_p_ratio])       #### dash level
cell_balance_box = wg.HBox([max_voltage_box, min_voltage_box, n_p_ratio_box])   #### dash level
# target_capacity = wg.IntSlider(value=9390,min=0,max=100000,step=50,continuous_update=False, layout=wg.Layout(width='80%'))
# start_capacity = wg.IntSlider(value=1220,min=0,max=30000,step=1,continuous_update=False, layout=wg.Layout(width='80%'))
target_capacity = wg.IntSlider(value=11934,min=0,max=20000,step=50,continuous_update=False, layout=wg.Layout(width='80%'))
start_capacity = wg.IntSlider(value=1215,min=0,max=2000,step=1,continuous_update=False, layout=wg.Layout(width='80%'))
target_capacity_box = wg.HBox([wg.Label(value='Reversible Capacity:'), target_capacity, wg.Label(value='mAh')])
start_capacity_box = wg.HBox([wg.Label(value='Irrev. Capacity Loss:'), start_capacity, wg.Label(value='mAh')])

def percentage_total(am1,am2,am3,ac1,ac2,ca1,ca2,bd1,bd2):
    total = round(am1+am2+am3+ac1+ac2+ca1+ca2+bd1+bd2,2)
    if round(total,2) != 100.00:
        response = '{0}% - Formulation Incomplete'.format(total)
    else:
        response = '{0}% - Formulation Valid'.format(total)
    print(response)
        

############## Anode Formulation Details ##############

# Anode Formulation
style = {'description_width': 'initial'}
a_am1_percentage = wg.FloatText(value=88, layout=wg.Layout(width='8%'))
a_am1_density = wg.FloatText(value=1.5, layout=wg.Layout(width='8%'))
a_am1_cost = wg.FloatText(value=14.27, layout=wg.Layout(width='10%'))
a_am1_percentage_box = wg.HBox([wg.Label(value='Active Material 1:', layout=wg.Layout(width='40%')),a_am1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_am1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_am1_cost, wg.Label(value='$/kg')])
a_am2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
a_am2_density = wg.FloatText(value=1.5, layout=wg.Layout(width='8%'))
a_am2_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
a_am2_percentage_box = wg.HBox([wg.Label(value='Active Material 2:', layout=wg.Layout(width='40%')),a_am2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_am2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_am2_cost, wg.Label(value='$/kg')])
a_am3_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
a_am3_density = wg.FloatText(value=1.5, layout=wg.Layout(width='8%'))
a_am3_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
a_am3_percentage_box = wg.HBox([wg.Label(value='Active Material 3:', layout=wg.Layout(width='40%')),a_am3_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_am3_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_am3_cost, wg.Label(value='$/kg')])
a_addcomp1_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
a_addcomp1_density = wg.FloatText(value=2.0, layout=wg.Layout(width='8%'))
a_addcomp1_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
a_addcomp1_percentage_box = wg.HBox([wg.Label(value='Additional Comp. 1:', layout=wg.Layout(width='40%')),a_addcomp1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_addcomp1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_addcomp1_cost, wg.Label(value='$/kg')])
a_addcomp2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
a_addcomp2_density = wg.FloatText(value=2.0, layout=wg.Layout(width='8%'))
a_addcomp2_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
a_addcomp2_percentage_box = wg.HBox([wg.Label(value='Additional Comp. 2:', layout=wg.Layout(width='40%')),a_addcomp2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_addcomp2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_addcomp2_cost, wg.Label(value='$/kg')])
a_condaid1_percentage = wg.FloatText(value=9, layout=wg.Layout(width='8%'))
a_condaid1_density = wg.FloatText(value=1.9, layout=wg.Layout(width='8%'))
a_condaid1_cost = wg.FloatText(value=9.0, layout=wg.Layout(width='10%'))
a_condaid1_percentage_box = wg.HBox([wg.Label(value='Conductive Aid 1:', layout=wg.Layout(width='40%')),a_condaid1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_condaid1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_condaid1_cost, wg.Label(value='$/kg')])
a_condaid2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
a_condaid2_density = wg.FloatText(value=1.9, layout=wg.Layout(width='8%'))
a_condaid2_cost = wg.FloatText(value=9.0, layout=wg.Layout(width='10%'))
a_condaid2_percentage_box = wg.HBox([wg.Label(value='Conductive Aid 2:', layout=wg.Layout(width='40%')),a_condaid2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_condaid2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_condaid2_cost, wg.Label(value='$/kg')])
a_binder1_percentage = wg.FloatText(value=3, layout=wg.Layout(width='8%'))
a_binder1_density = wg.FloatText(value=1.7, layout=wg.Layout(width='8%'))
a_binder1_cost = wg.FloatText(value=10.0, layout=wg.Layout(width='10%'))
a_binder1_percentage_box = wg.HBox([wg.Label(value='Binder 1:', layout=wg.Layout(width='40%')),a_binder1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_binder1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_binder1_cost, wg.Label(value='$/kg')])
a_binder2_percentage = wg.FloatText(value=0, layout=wg.Layout(width='8%'))
a_binder2_density = wg.FloatText(value=1.7, layout=wg.Layout(width='8%'))
a_binder2_cost = wg.FloatText(value=10.0, layout=wg.Layout(width='10%'))
a_binder2_percentage_box = wg.HBox([wg.Label(value='Binder 2:', layout=wg.Layout(width='40%')),a_binder2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                a_binder2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                a_binder2_cost, wg.Label(value='$/kg')])

a_density = wg.FloatText(value=0.85, layout=wg.Layout(width='8%'))
a_density_box = wg.HBox([wg.Label(value='Calender Density:', layout=wg.Layout(width='40%')),a_density,wg.Label(value='g/cc')])

a_out = wg.interactive_output(percentage_total, {'am1':a_am1_percentage,
                                                  'am2':a_am2_percentage,
                                                  'am3':a_am3_percentage,
                                                  'ac1':a_addcomp1_percentage,
                                                  'ac2':a_addcomp2_percentage,
                                                  'ca1':a_condaid1_percentage,
                                                  'ca2':a_condaid2_percentage,
                                                  'bd1':a_binder1_percentage,
                                                  'bd2':a_binder2_percentage,
                                                 })

a_porosity = wg.Label(value='1.0', layout=wg.Layout(width='12%'))
a_porosity_box = wg.HBox([wg.Label(value='Porosity:', layout=wg.Layout(width='40%')), a_porosity, wg.Label(value='%')])


anodeFormulationDetails = wg.VBox([a_am1_percentage_box,
                                   a_am2_percentage_box,
                                   a_am3_percentage_box,
                                   a_addcomp1_percentage_box,
                                   a_addcomp2_percentage_box,
                                   a_condaid1_percentage_box,
                                   a_condaid2_percentage_box,
                                   a_binder1_percentage_box,
                                   a_binder2_percentage_box,
                                   a_out,
                                   a_density_box,
                                   a_porosity_box
                                  ], margin='10px 10px 10px 10px')

# Anode Materials Selection
anode_am1_value = anode_options[0]
anode_am2_value = anode_options[1]
anode_am3_value = anode_options[2]
anode_am1 = wg.Dropdown(options=anode_options, value=anode_am1_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
anode_am1_dropdown_box = wg.HBox([wg.Label(value='Active Material 1:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), anode_am1])
anode_am1_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am1_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am1_irrev_scale])
anode_am1_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am1_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am1_rev_scale])
anode_am2 = wg.Dropdown(options=anode_options, value=anode_am2_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
anode_am2_dropdown_box = wg.HBox([wg.Label(value='Active Material 2:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), anode_am2])
anode_am2_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am2_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am2_irrev_scale])
anode_am2_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am2_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am2_rev_scale])
anode_am3 = wg.Dropdown(options=anode_options, value=anode_am3_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
anode_am3_dropdown_box = wg.HBox([wg.Label(value='Active Material 3:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), anode_am3])
anode_am3_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am3_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am3_irrev_scale])
anode_am3_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_am3_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), anode_am3_rev_scale])

anode_materials = wg.VBox([anode_am1_dropdown_box, anode_am1_irrev_box, anode_am1_rev_box, 
                           anode_am2_dropdown_box, anode_am2_irrev_box, anode_am2_rev_box,
                           anode_am3_dropdown_box, anode_am3_irrev_box, anode_am3_rev_box,
                           ], layout=wg.Layout(width='35%'))


############## Cathode Formulation Details ##############
# Cathode Formulation
c_am1_percentage = wg.FloatText(value=89.0, layout=wg.Layout(width='8%'))
c_am1_density = wg.FloatText(value=4, layout=wg.Layout(width='8%'))
c_am1_cost = wg.FloatText(value=11.26, layout=wg.Layout(width='10%'))
c_am1_percentage_box = wg.HBox([wg.Label(value='Active Material 1:', layout=wg.Layout(width='40%')),c_am1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_am1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_am1_cost, wg.Label(value='$/kg')])
c_am2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_am2_density = wg.FloatText(value=4, layout=wg.Layout(width='8%'))
c_am2_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
c_am2_percentage_box = wg.HBox([wg.Label(value='Active Material 2:', layout=wg.Layout(width='40%')),c_am2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_am2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_am2_cost, wg.Label(value='$/kg')])
c_am3_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_am3_density = wg.FloatText(value=4, layout=wg.Layout(width='8%'))
c_am3_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
c_am3_percentage_box = wg.HBox([wg.Label(value='Active Material 3:', layout=wg.Layout(width='40%')),c_am3_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_am3_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_am3_cost, wg.Label(value='$/kg')])
c_addcomp1_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_addcomp1_density = wg.FloatText(value=2.0, layout=wg.Layout(width='8%'))
c_addcomp1_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
c_addcomp1_percentage_box = wg.HBox([wg.Label(value='Additional Comp. 1:', layout=wg.Layout(width='40%')),c_addcomp1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_addcomp1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_addcomp1_cost, wg.Label(value='$/kg')])
c_addcomp2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_addcomp2_density = wg.FloatText(value=2.0, layout=wg.Layout(width='8%'))
c_addcomp2_cost = wg.FloatText(value=0.0, layout=wg.Layout(width='10%'))
c_addcomp2_percentage_box = wg.HBox([wg.Label(value='Additional Comp. 2:', layout=wg.Layout(width='40%')),c_addcomp2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_addcomp2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_addcomp2_cost, wg.Label(value='$/kg')])
c_condaid1_percentage = wg.FloatText(value=6.0, layout=wg.Layout(width='8%'))
c_condaid1_density = wg.FloatText(value=1.9, layout=wg.Layout(width='8%'))
c_condaid1_cost = wg.FloatText(value=9.0, layout=wg.Layout(width='10%'))
c_condaid1_percentage_box = wg.HBox([wg.Label(value='Conductive Aid 1:', layout=wg.Layout(width='40%')),c_condaid1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_condaid1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_condaid1_cost, wg.Label(value='$/kg')])
c_condaid2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_condaid2_density = wg.FloatText(value=1.9, layout=wg.Layout(width='8%'))
c_condaid2_cost = wg.FloatText(value=9.0, layout=wg.Layout(width='10%'))
c_condaid2_percentage_box = wg.HBox([wg.Label(value='Conductive Aid 2:', layout=wg.Layout(width='40%')),c_condaid2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_condaid2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_condaid2_cost, wg.Label(value='$/kg')])
c_binder1_percentage = wg.FloatText(value=5.0, layout=wg.Layout(width='8%'))
c_binder1_density = wg.FloatText(value=1.7, layout=wg.Layout(width='8%'))
c_binder1_cost = wg.FloatText(value=15.0, layout=wg.Layout(width='10%'))
c_binder1_percentage_box = wg.HBox([wg.Label(value='Binder 1:', layout=wg.Layout(width='40%')),c_binder1_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_binder1_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_binder1_cost, wg.Label(value='$/kg')])
c_binder2_percentage = wg.FloatText(value=0.0, layout=wg.Layout(width='8%'))
c_binder2_density = wg.FloatText(value=1.7, layout=wg.Layout(width='8%'))
c_binder2_cost = wg.FloatText(value=15.0, layout=wg.Layout(width='10%'))
c_binder2_percentage_box = wg.HBox([wg.Label(value='Binder 2:', layout=wg.Layout(width='40%')),c_binder2_percentage,wg.Label(value='%', layout=wg.Layout(width='5%')),
                                c_binder2_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
                                c_binder2_cost, wg.Label(value='$/kg')])

c_density = wg.FloatText(value=2.6, layout=wg.Layout(width='8%'))
c_density_box = wg.HBox([wg.Label(value='Calender Density:', layout=wg.Layout(width='40%')),c_density,wg.Label(value='g/cc')])

c_out = wg.interactive_output(percentage_total, {'am1':c_am1_percentage,
                                                  'am2':c_am2_percentage,
                                                  'am3':c_am3_percentage,
                                                  'ac1':c_addcomp1_percentage,
                                                  'ac2':c_addcomp2_percentage,
                                                  'ca1':c_condaid1_percentage,
                                                  'ca2':c_condaid2_percentage,
                                                  'bd1':c_binder1_percentage,
                                                  'bd2':c_binder2_percentage,
                                                 })

c_porosity = wg.Label(value='1.0', layout=wg.Layout(width='12%'))
c_porosity_box = wg.HBox([wg.Label(value='Porosity:', layout=wg.Layout(width='45%')), c_porosity, wg.Label(value='%')])

cathodeFormulationDetails = wg.VBox([c_am1_percentage_box,
                                     c_am2_percentage_box,
                                     c_am3_percentage_box,
                                     c_addcomp1_percentage_box,
                                     c_addcomp2_percentage_box,
                                     c_condaid1_percentage_box,
                                     c_condaid2_percentage_box,
                                     c_binder1_percentage_box,
                                     c_binder2_percentage_box,
                                     c_out,
                                     c_density_box,
                                     c_porosity_box
                                     ], margin='10px 10px 10px 10px')

# Cathode Materials Selection
cathode_am1_value = cathode_options[0]
cathode_am2_value = cathode_options[1]
cathode_am3_value = cathode_options[2]
cathode_am1 = wg.Dropdown(options=cathode_options, value=cathode_am1_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
cathode_am1_dropdown_box = wg.HBox([wg.Label(value='Active Material 1:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), cathode_am1])
cathode_am1_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am1_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am1_irrev_scale])
cathode_am1_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am1_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am1_rev_scale])
cathode_am2 = wg.Dropdown(options=cathode_options, value=cathode_am2_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
cathode_am2_dropdown_box = wg.HBox([wg.Label(value='Active Material 2:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), cathode_am2])
cathode_am2_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am2_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am2_irrev_scale])
cathode_am2_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am2_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am2_rev_scale])
cathode_am3 = wg.Dropdown(options=cathode_options, value=cathode_am3_value, layout=wg.Layout(margin='15px 0px 0px 0px'))
cathode_am3_dropdown_box = wg.HBox([wg.Label(value='Active Material 3:', layout=wg.Layout(width='25%', margin='15px 0px 0px 0px')), cathode_am3])
cathode_am3_irrev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am3_irrev_box = wg.HBox([wg.Label(value='Irrev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am3_irrev_scale])
cathode_am3_rev_scale = wg.FloatSlider(value=1.00, min=0.7, 
                                 max=1.3,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
cathode_am3_rev_box = wg.HBox([wg.Label(value='Rev. Cap Scaling:', layout=wg.Layout(width='25%')), cathode_am3_rev_scale])

cathode_materials = wg.VBox([cathode_am1_dropdown_box, cathode_am1_irrev_box, cathode_am1_rev_box,
                             cathode_am2_dropdown_box, cathode_am2_irrev_box, cathode_am2_rev_box,
                             cathode_am3_dropdown_box, cathode_am3_irrev_box, cathode_am3_rev_box,
                             ], layout=wg.Layout(width='35%'))



############## Electrode Balancing Details ##############

style = {'description_width': 'initial'}
anode_ML_slider = wg.FloatSlider(value=5.25, 
                                 min=0.0, 
                                 max=35.00,
                                 step=0.01,
                                 readout_format='.2f',
                                 layout=wg.Layout(flex='3 1 0%', width='auto'),
                                 continuous_update=False)
anode_ML_slider_box = wg.HBox([wg.Label(value='Anode Mass Loading (mg/cm2):'), anode_ML_slider])

cathode_ML_slider = wg.FloatSlider(value=10.68,
                                   min=1.0,
                                   max=38.00,
                                   step=0.01,
                                   readout_format='.2f',
                                   layout=wg.Layout(flex='3 1 0%', width='auto'),
                                   continuous_update=False)
cathode_ML_slider_box = wg.HBox([wg.Label(value='Cathode Mass Loading (mg/cm2):'), cathode_ML_slider])

cathode_stacks_slider = wg.IntSlider(value=26,min=1,max=110,step=1)
cathode_stacks_slider_box = wg.HBox([wg.Label(value='# Cathode Stacks:'), cathode_stacks_slider])

c_area = wg.FloatText(value=172.83, min=0.1, layout=wg.Layout(width='35%'))
c_area_box = wg.HBox([wg.Label(value='Cathode Single-sided Area:'), c_area, wg.Label(value='cm2')])

overhang = wg.FloatSlider(value=0.00, min=0.0, max = 20.0)
overhang_box = wg.HBox([wg.Label(value='Anode Overhang %:'),overhang])

c_ss_thickness = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
c_ds_thickness = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
c_areal_cap = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
c_ss_thickness_box = wg.HBox([wg.Label(value='Cathode SS Thickness:', layout=wg.Layout(width='12%')), c_ss_thickness, wg.Label(value='µm', layout=wg.Layout(margin='0px 50px 0px 0px')),
                              wg.Label(value='Cathode DS Thickness:', layout=wg.Layout(width='12%')), c_ds_thickness, wg.Label(value='µm', layout=wg.Layout(margin='0px 50px 0px 0px')),
                              wg.Label(value='Cathode Areal Capacity:', layout=wg.Layout(width='12%')), c_areal_cap, wg.Label(value='mAh/cm2')])
a_ss_thickness = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
a_ds_thickness = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
eff_areal_cap = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
a_ss_thickness_box = wg.HBox([wg.Label(value='Anode SS Thickness:', layout=wg.Layout(width='12%')), a_ss_thickness, wg.Label(value='µm', layout=wg.Layout(margin='0px 50px 0px 0px')),
                              wg.Label(value='Anode DS Thickness:', layout=wg.Layout(width='12%')), a_ds_thickness, wg.Label(value='µm', layout=wg.Layout(margin='0px 50px 0px 0px')),
                              wg.Label(value='Effective Areal Capacity:', layout=wg.Layout(width='12%')), eff_areal_cap, wg.Label(value='mAh/cm2')])



############## Other Cell Design Inputs ##############

cathode_coated_area = wg.FloatText(value='1.0', disabled=True, layout=wg.Layout(width='5%'))
cathode_uncoated_area = wg.FloatText(value=8.22, layout=wg.Layout(width='5%'))
cathode_cc_thickness = wg.FloatText(value=15.0, layout=wg.Layout(width='5%'))
cathode_cc_density = wg.FloatText(value=2.7, layout=wg.Layout(width='5%'))
cathode_cc_cost = wg.FloatText(value=6.3, layout=wg.Layout(width='5%'))
cathode_foil_area_box = wg.HBox([
    wg.Label(value='Coated Area:', ), cathode_coated_area, wg.Label(value='cm2', layout=wg.Layout(width='5%')),
    wg.Label(value='Bare Tab Area:', ), cathode_uncoated_area, wg.Label(value='cm2', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Thickness:', ), cathode_cc_thickness, wg.Label(value='µm', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Density:', ), cathode_cc_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Cost:',), cathode_cc_cost, wg.Label(value='$/kg', layout=wg.Layout(width='5%')), 
])
cathode_details_box = wg.VBox([wg.Label(value='CATHODE DETAILS'), cathode_foil_area_box])

anode_coated_area = wg.FloatText(value=181.2, layout=wg.Layout(width='5%'))
anode_uncoated_area = wg.FloatText(value=7.55, layout=wg.Layout(width='5%'))
anode_cc_thickness = wg.FloatText(value=15.0, layout=wg.Layout(width='5%'))
anode_cc_density = wg.FloatText(value=2.7, layout=wg.Layout(width='5%'))
anode_cc_cost = wg.FloatText(value=6.3, layout=wg.Layout(width='5%'))
anode_foil_area_box = wg.HBox([
    wg.Label(value='Coated Area:', ), anode_coated_area, wg.Label(value='cm2', layout=wg.Layout(width='5%')),
    wg.Label(value='Bare Tab Area:', ), anode_uncoated_area, wg.Label(value='cm2', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Thickness:', ), anode_cc_thickness, wg.Label(value='µm', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Density:', ), anode_cc_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
    wg.Label(value='CC Cost:',), anode_cc_cost, wg.Label(value='$/kg', layout=wg.Layout(width='5%')), 
])
anode_details_box = wg.VBox([wg.Label(value='ANODE DETAILS'), anode_foil_area_box])

separator_thickness = wg.FloatText(value=16.0, layout=wg.Layout(width='5%'))
separator_width = wg.FloatText(value=100, layout=wg.Layout(width='5%'))
separator_fold_length = wg.FloatText(value=186, layout=wg.Layout(width='5%'))
separator_density = wg.FloatText(value=0.4, layout=wg.Layout(width='5%'))
separator_porosity = wg.FloatText(value=47, layout=wg.Layout(width='5%'))
separator_cost = wg.FloatText(value=0.9, layout=wg.Layout(width='5%'))
separator_film_box_1 = wg.HBox([
    wg.Label(value='Thickness:', ), separator_thickness, wg.Label(value='µm', layout=wg.Layout(width='5%')),
    wg.Label(value='Slit Width:', ), separator_width, wg.Label(value='mm', layout=wg.Layout(width='5%')),
    wg.Label(value='Fold Length:', ), separator_fold_length, wg.Label(value='mm', layout=wg.Layout(width='5%')),
])
separator_film_box_2 = wg.HBox([
    wg.Label(value='Sep. Density:', ), separator_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
    wg.Label(value='Sep. Porosity:', ), separator_porosity, wg.Label(value='%', layout=wg.Layout(width='5%')),
    wg.Label(value='Sep. Cost:',), separator_cost, wg.Label(value='$/m2', layout=wg.Layout(width='5%')), 
])
separator_details_box = wg.VBox([wg.Label(value='SEPARATOR DETAILS'), separator_film_box_1, separator_film_box_2])


elyte_overhead = wg.FloatText(value=10.0, layout=wg.Layout(width='8%'))
elyte_density = wg.FloatText(value=1.2, layout=wg.Layout(width='8%'))
elyte_cost = wg.FloatText(value=8.94, layout=wg.Layout(width='5%'))
elyte_norm_amount = wg.Label(value='1.0', layout=wg.Layout(margin='0px 10px 0px 0px'))
elyte_box = wg.HBox([
    wg.Label(value='Density:', ), elyte_density, wg.Label(value='g/cc', layout=wg.Layout(width='5%')),
    wg.Label(value='Overfill:', ), elyte_overhead, wg.Label(value='%', layout=wg.Layout(width='5%')),
    wg.Label(value='Ely. Cost:',), elyte_cost, wg.Label(value='$/kg', layout=wg.Layout(width='5%')), 
    wg.Label(value='Norm. Amount:', ), elyte_norm_amount, wg.Label(value='mL/Ah', layout=wg.Layout(margin='0px 5px 0px 0px')),
])
elyte_details_box = wg.VBox([wg.Label(value='ELECTROLYTE DETAILS'), elyte_box])

c_swell_factor = wg.FloatText(value=1.0, layout=wg.Layout(width='8%'))
a_swell_factor = wg.FloatText(value=1.0, layout=wg.Layout(width='8%'))
swell_box = wg.HBox([
    wg.Label(value='Cathode Swell Factor:', ), c_swell_factor,
    wg.Label(value='Anode Swell Factor:',), a_swell_factor,
])
swell_details_box = wg.VBox([wg.Label(value='SWELLING CHARACTERISTICS'), swell_box])
elyte_swell_box = wg.VBox([elyte_details_box, swell_details_box])


mylar_thickness = wg.FloatText(value=113.0, layout=wg.Layout(width='5%'))
mylar_areal_mass = wg.FloatText(value=18.0, layout=wg.Layout(width='5%'))
mylar_cost = wg.FloatText(value=4.64, layout=wg.Layout(width='5%'))
pos_terminal_mass = wg.FloatText(value=1.0, layout=wg.Layout(width='5%'))
pos_terminal_cost = wg.FloatText(value=16.0, layout=wg.Layout(width='5%'))
neg_terminal_mass = wg.FloatText(value=1.0, layout=wg.Layout(width='5%'))
neg_terminal_cost = wg.FloatText(value=16.0, layout=wg.Layout(width='5%'))
other_mass = wg.FloatText(value=0.3, layout=wg.Layout(width='5%'))
seal_1_length = wg.FloatText(value=22, layout=wg.Layout(width='5%'))
seal_2_length = wg.FloatText(value=7, layout=wg.Layout(width='5%'))
seal_3_length = wg.FloatText(value=7, layout=wg.Layout(width='5%'))
seal_4_length = wg.FloatText(value=0, layout=wg.Layout(width='5%'))
pouch_length = wg.FloatText(value=188, layout=wg.Layout(width='5%'))
pouch_width = wg.FloatText(value=102.5, layout=wg.Layout(width='5%'))
laminate_box = wg.HBox([
    wg.Label(value='Laminate Thickness:', ), mylar_thickness, wg.Label(value='µm', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Laminate Areal Mass:',), mylar_areal_mass, wg.Label(value='mg/cm2', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Laminate Cost:',), mylar_cost, wg.Label(value='$/m2', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Tape/Other Mass:', ), other_mass, wg.Label(value='g', layout=wg.Layout(margin='0px 20px 0px 0px')),
])
other_mass_box = wg.HBox([
    wg.Label(value='Positive Terminal Mass:',), pos_terminal_mass, wg.Label(value='g', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Positive Terminal Cost:',), pos_terminal_cost, wg.Label(value='$/kg', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Negative Terminal Mass:', ), neg_terminal_mass, wg.Label(value='g', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Negative Terminal Cost:',), neg_terminal_cost, wg.Label(value='$/kg', layout=wg.Layout(margin='0px 30px 0px 0px')),
])
sealing_box = wg.HBox([
    wg.Label(value='Seal/Terrace 1 Length:', ), seal_1_length, wg.Label(value='mm', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Seal/Terrace 2 Length:', ), seal_2_length, wg.Label(value='mm', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Seal/Terrace 3 Length:', ), seal_3_length, wg.Label(value='mm', layout=wg.Layout(margin='0px 30px 0px 0px')),
    wg.Label(value='Seal/Terrace 4 Length:', ), seal_4_length, wg.Label(value='mm', layout=wg.Layout(margin='0px 30px 0px 0px')),    
])
pouch_dim_box = wg.HBox([wg.Label(value='Pouch Cup Dim. (lxw):',), pouch_length, wg.Label(value='x'), pouch_width, wg.Label(value='mm', layout=wg.Layout(margin='0px 20px 0px 0px')),])
inactive_details_box = wg.VBox([wg.Label(value='INACTIVE MATERIAL DETAILS'), laminate_box, other_mass_box, sealing_box, pouch_dim_box])


############## Final Cell Design Outputs ##############

cell_energy = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
cell_mass = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
cell_thickness = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
specific_energy = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
energy_density = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
cell_cost = wg.Label(value='1.0', layout=wg.Layout(width='5%'))
cell_normalized_cost = wg.Label(value='1.0', layout=wg.Layout(width='5%'))

cell_energy_box = wg.HBox([
    wg.Label(value='Cell Energy:', layout=wg.Layout(width='10%')), cell_energy, wg.Label(value='Wh')
])
cell_mass_box = wg.HBox([
    wg.Label(value='Cell Mass:', layout=wg.Layout(width='10%')), cell_mass, wg.Label(value='g')
])
cell_thickness_box = wg.HBox([
    wg.Label(value='Cell Thickness:', layout=wg.Layout(width='10%')), cell_thickness, wg.Label(value='mm')
])
cell_specific_energy_box = wg.HBox([
    wg.Label(value='Specific Energy:', layout=wg.Layout(width='10%')), specific_energy, wg.Label(value='Wh/kg')
])
cell_energy_density_box = wg.HBox([
    wg.Label(value='Energy Density:', layout=wg.Layout(width='10%')), energy_density, wg.Label(value='Wh/L')
])
cell_cost_box = wg.HBox([
    wg.Label(value='Cell Cost:', layout=wg.Layout(width='10%')), wg.Label(value='$'), cell_cost 
])
cell_normalized_cost_box = wg.HBox([
    wg.Label(value='Normalized Cost:', layout=wg.Layout(width='10%')), wg.Label(value='$'), cell_normalized_cost, wg.Label(value='/kWh')
])

### Open Circuit Voltages
soc_0 = wg.Label(value='1.0')
soc_10 = wg.Label(value='1.0')
soc_20 = wg.Label(value='1.0')
soc_30 = wg.Label(value='1.0')
soc_40 = wg.Label(value='1.0')
soc_50 = wg.Label(value='1.0')
soc_60 = wg.Label(value='1.0')
soc_70 = wg.Label(value='1.0')
soc_80 = wg.Label(value='1.0')
soc_90 = wg.Label(value='1.0')
soc_100 = wg.Label(value='1.0')

positive_voltage_vals = wg.Label(value='[]')
positive_capacity_vals = wg.Label(value='[]')
negative_voltage_vals = wg.Label(value='[]')
negative_capacity_vals = wg.Label(value='[]')

full_voltage_vals1 = wg.Label(value='[]')
full_capacity_vals1 = wg.Label(value='[]')
full_voltage_vals2 = wg.Label(value='[]')
full_capacity_vals2 = wg.Label(value='[]')


############## Matching Curves ##############

matching_curve_file = wg.Text('')
matching_curve_grams = wg.IntSlider(value=0 ,min=0,max=1500,step=1,continuous_update=False)



######## Plotting ########
def plot_main_fig(u_max, u_min, n_p_ratio, 
                  target_capacity, start_capacity, 
                  anode_am1, anode_am2, anode_am3,
                  anode_am1_irrev_scale, anode_am1_rev_scale,
                  anode_am2_irrev_scale, anode_am2_rev_scale,
                  anode_am3_irrev_scale, anode_am3_rev_scale,
                  a_am1, a_am2, a_am3, a_ac1, a_ac2, a_ca1, a_ca2, a_bd1, a_bd2,
                  a_am1_density, a_am2_density, a_am3_density, a_addcomp1_density, a_addcomp2_density, 
                  a_condaid1_density, a_condaid2_density, a_binder1_density, a_binder2_density,
                  a_density,
                  cathode_am1, cathode_am2, cathode_am3,
                  cathode_am1_irrev_scale, cathode_am1_rev_scale,                  
                  cathode_am2_irrev_scale, cathode_am2_rev_scale,                  
                  cathode_am3_irrev_scale, cathode_am3_rev_scale,                  
                  c_am1, c_am2, c_am3, c_ac1, c_ac2, c_ca1, c_ca2, c_bd1, c_bd2,
                  c_am1_density, c_am2_density, c_am3_density, c_addcomp1_density, c_addcomp2_density, 
                  c_condaid1_density, c_condaid2_density, c_binder1_density, c_binder2_density,
                  c_density,
                  anode_ML, cathode_ML, cathode_stacks, c_area, overhang,
                  matching_curve_file, matching_curve_grams,
                  cathode_uncoated_area, cathode_cc_thickness, cathode_cc_density,
                  anode_coated_area, anode_uncoated_area, anode_cc_thickness, anode_cc_density,
                  separator_thickness, separator_width, separator_fold_length, separator_density, separator_porosity,
                  elyte_overhead, elyte_density, c_swell_factor, a_swell_factor,
                  mylar_thickness, mylar_areal_mass, pos_terminal_mass, neg_terminal_mass, other_mass, 
                  seal_1_length, seal_2_length, seal_3_length, seal_4_length, pouch_length, pouch_width,
                  a_am1_cost, a_am2_cost, a_am3_cost, a_addcomp1_cost, a_addcomp2_cost, a_condaid1_cost, a_condaid2_cost, a_binder1_cost, a_binder2_cost,
                  c_am1_cost, c_am2_cost, c_am3_cost, c_addcomp1_cost, c_addcomp2_cost, c_condaid1_cost, c_condaid2_cost, c_binder1_cost, c_binder2_cost,
                  cathode_cc_cost, anode_cc_cost, separator_cost, elyte_cost, mylar_cost, pos_terminal_cost, neg_terminal_cost
                 ):
    
    ####### Performing Calculations ######## 
    
    cathode_coated_area.value = str(c_area)
    
    def porosity_calculator(am1, am1_d, am2, am2_d, am3, am3_d, ac1, ac1_d, ac2, ac2_d, ca1, ca1_d, ca2, ca2_d, bd1, bd1_d, bd2, bd2_d, calender_density):
        try:
            theo_specific_vol = am1*(1/am1_d)/100 + am2*(1/am2_d)/100 + am3*(1/am3_d)/100 + ac1*(1/ac1_d)/100 + ac2*(1/ac2_d)/100 + ca1*(1/ca1_d)/100 + ca2*(1/ca2_d)/100 + bd1*(1/bd1_d)/100 + bd2*(1/bd2_d)/100
            porosity = (1 - (theo_specific_vol*calender_density)) * 100
        except:
            porosity = 'Error: Densities must be non-zero'
        return porosity
    
    anode_porosity = porosity_calculator(a_am1, a_am1_density, a_am2, a_am2_density, a_am3, a_am3_density,
                                         a_ac1, a_addcomp1_density, a_ac2, a_addcomp2_density,
                                         a_ca1, a_condaid1_density, a_ca2, a_condaid2_density,
                                         a_bd1, a_binder1_density, a_bd2, a_binder2_density,
                                         a_density
                                        )
    a_porosity.value = str(round(anode_porosity, 1))
    
    cathode_porosity = porosity_calculator(c_am1, c_am1_density, c_am2, c_am2_density, c_am3, c_am3_density,
                                         c_ac1, c_addcomp1_density, c_ac2, c_addcomp2_density,
                                         c_ca1, c_condaid1_density, c_ca2, c_condaid2_density,
                                         c_bd1, c_binder1_density, c_bd2, c_binder2_density,
                                         c_density
                                        )
    c_porosity.value = str(round(cathode_porosity, 1))
    
    anode_ss_thickness = anode_ML / a_density / 1000 * 10000 # units in µm
    anode_ds_thickness = anode_ss_thickness*2 + anode_cc_thickness
    a_ss_thickness.value = str(round(anode_ss_thickness, 1))
    a_ds_thickness.value = str(round(anode_ds_thickness, 1))
    
    cathode_ss_thickness = cathode_ML / c_density / 1000 * 10000 # units in µm
    cathode_ds_thickness = cathode_ss_thickness*2 + cathode_cc_thickness
    c_ss_thickness.value = str(round(cathode_ss_thickness, 1))
    c_ds_thickness.value = str(round(cathode_ds_thickness, 1))
    
    active_geo_area = c_area * 2 * cathode_stacks
    effective_areal_cap = target_capacity / active_geo_area
    eff_areal_cap.value = str(round(effective_areal_cap, 2))
    
    
    
    cathode_coat_mass_per_sheet = c_area * cathode_ML / 1000 * 2
    cathode_foil_mass_per_sheet = (c_area + cathode_uncoated_area) * cathode_cc_thickness/10000 * cathode_cc_density
    total_cathode_cc_cost = (cathode_foil_mass_per_sheet * cathode_stacks) / 1000 * cathode_cc_cost # in $
    total_cathode_mass = (cathode_coat_mass_per_sheet + cathode_foil_mass_per_sheet) * cathode_stacks
    
    anode_coat_mass_per_sheet = anode_coated_area * anode_ML / 1000 * 2
    anode_foil_mass_per_sheet = (anode_coated_area + anode_uncoated_area) * anode_cc_thickness/10000 * anode_cc_density
    total_anode_cc_cost = (anode_foil_mass_per_sheet * (cathode_stacks+1)) / 1000 * anode_cc_cost # in $
    total_anode_mass = (anode_coat_mass_per_sheet + anode_foil_mass_per_sheet) * (cathode_stacks+1)
    
    total_separator_area = (separator_width/10 * separator_fold_length/10) * (cathode_stacks*2 + 3) # cm2
    total_separator_mass = (separator_thickness/10000 * total_separator_area) * separator_density
    total_separator_cost = total_separator_area/10000 * separator_cost # in $
    
    total_cathode_pore_volume = (c_area * cathode_ss_thickness/10000 * c_swell_factor * 2 * cathode_stacks) * cathode_porosity/100
    total_anode_pore_volume = (anode_coated_area * anode_ss_thickness/10000 * a_swell_factor * 2 * (cathode_stacks+1)) * anode_porosity/100
    total_separator_pore_volume = (separator_thickness/10000 * (separator_width/10 * separator_fold_length/10)) * (cathode_stacks*2 + 3) * separator_porosity/100
    elyte_volume_in_pores = total_cathode_pore_volume + total_anode_pore_volume + total_separator_pore_volume
    
    total_elyte_volume = elyte_volume_in_pores * (1+elyte_overhead/100)
    elyte_norm_amount.value = str(round(total_elyte_volume / (target_capacity/1000), 2))
    total_elyte_mass = total_elyte_volume * elyte_density
    total_elyte_cost = total_elyte_mass/1000 * elyte_cost # in $
    
    total_pouch_length = pouch_length + seal_1_length + seal_4_length
    total_pouch_width = pouch_width + seal_2_length + seal_3_length
    total_pouch_area = total_pouch_length/10 * total_pouch_width/10
    total_pouch_cost = total_pouch_area/10000 * mylar_cost # in $
    total_pouch_mass = (2 * total_pouch_area * mylar_areal_mass)/1000

    
    total_pos_term_cost = pos_terminal_mass/1000 * pos_terminal_cost # in $
    total_neg_term_cost = neg_terminal_mass/1000 * neg_terminal_cost # in $
    
    total_cell_mass = total_cathode_mass + total_anode_mass + total_separator_mass + total_elyte_mass + total_pouch_mass + pos_terminal_mass + neg_terminal_mass + other_mass # in g
    cell_mass.value = str(round(total_cell_mass, 2))
    
    total_cathode_thickness = ((cathode_ss_thickness/1000 * c_swell_factor * 2) + cathode_cc_thickness/1000) * cathode_stacks # in mm
    total_anode_thickness = ((anode_ss_thickness/1000 * a_swell_factor * 2) + anode_cc_thickness/1000) * (cathode_stacks+1) # in mm
    total_separator_thickness = (separator_thickness * (cathode_stacks*2 + 3)) / 1000 # in mm
    total_mylar_thickness = (mylar_thickness * 2)/1000
    
    total_cell_thickness = total_cathode_thickness + total_anode_thickness + total_separator_thickness + total_mylar_thickness
    cell_thickness.value = str(round(total_cell_thickness, 2))
    total_cell_volume = (total_cell_thickness/10 * (total_pouch_length)/10 * (total_pouch_width)/10)/1000 # in Liters
    
    fig, host = plt.subplots(figsize=(12,7))
    
    ############## Plotting Theoretical Cathode Half-Cell Curves ##############
    
    c_am1_grams = (c_am1/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_am2_grams = (c_am2/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_am3_grams = (c_am3/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_addcomp1_grams = (c_ac1/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_addcomp2_grams = (c_ac2/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_condaid1_grams = (c_ca1/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_condaid2_grams = (c_ca2/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_binder1_grams = (c_bd1/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    c_binder2_grams = (c_bd2/100) * cathode_ML * c_area * 2 * cathode_stacks / 1000
    
    total_c_am1_cost = c_am1_grams/1000 * c_am1_cost # in $
    total_c_am2_cost = c_am2_grams/1000 * c_am2_cost # in $
    total_c_am3_cost = c_am3_grams/1000 * c_am3_cost # in $
    total_c_addcomp1_cost = c_addcomp1_grams/1000 * c_addcomp1_cost # in $
    total_c_addcomp2_cost = c_addcomp2_grams/1000 * c_addcomp2_cost # in $
    total_c_condaid1_cost = c_condaid1_grams/1000 * c_condaid1_cost # in $
    total_c_condaid2_cost = c_condaid2_grams/1000 * c_condaid2_cost # in $
    total_c_binder1_cost = c_binder1_grams/1000 * c_binder1_cost # in $
    total_c_binder2_cost = c_binder2_grams/1000 * c_binder2_cost # in $
        
    #Cathode AM1 Capacity
    cathode_am1_curve = pd.read_csv(os.path.join(half_cell_dir, 'Cathode_{0}.csv'.format(cathode_am1)))
    cathode_am1_curve['Absolute Capacity (mAh)'] = cathode_am1_curve['Specific Capacity (mAh/g)']*c_am1_grams
    #First charge -- Step 1
    y_interpCatCap11 = np.arange(0.5,5.0,0.001)
    x_interpCatCap11 = np.interp(y_interpCatCap11, cathode_am1_curve.loc[cathode_am1_curve['Step_ID']==1]['Voltage (V)'], cathode_am1_curve.loc[cathode_am1_curve['Step_ID']==1]['Absolute Capacity (mAh)'] * cathode_am1_irrev_scale)
    interpCatCap11 = pd.DataFrame({'Capacity':x_interpCatCap11, 'Voltage':y_interpCatCap11})
    #First discharge -- Step 2
    y_interpCatCap12 = np.arange(0.5,5.0,0.001)
    x_interpCatCap12 = np.interp(y_interpCatCap12, cathode_am1_curve.loc[cathode_am1_curve['Step_ID']==2]['Voltage (V)'][::-1], cathode_am1_curve.loc[cathode_am1_curve['Step_ID']==2]['Absolute Capacity (mAh)'][::-1] * cathode_am1_irrev_scale * cathode_am1_rev_scale)
    interpCatCap12 = pd.DataFrame({'Capacity':x_interpCatCap12, 'Voltage':y_interpCatCap12})

    #Cathode AM2 Capacity
    cathode_am2_curve = pd.read_csv(os.path.join(half_cell_dir, 'Cathode_{0}.csv'.format(cathode_am2)))
    cathode_am2_curve['Absolute Capacity (mAh)'] = cathode_am2_curve['Specific Capacity (mAh/g)']*c_am2_grams
    #First charge -- Step 1
    y_interpCatCap21 = np.arange(0.5,5.0,0.001)
    x_interpCatCap21 = np.interp(y_interpCatCap21, cathode_am2_curve.loc[cathode_am2_curve['Step_ID']==1]['Voltage (V)'], cathode_am2_curve.loc[cathode_am2_curve['Step_ID']==1]['Absolute Capacity (mAh)'] * cathode_am2_irrev_scale)
    interpCatCap21 = pd.DataFrame({'Capacity':x_interpCatCap21, 'Voltage':y_interpCatCap21})
    #First discharge -- Step 2
    y_interpCatCap22 = np.arange(0.5,5.0,0.001)
    x_interpCatCap22 = np.interp(y_interpCatCap22, cathode_am2_curve.loc[cathode_am2_curve['Step_ID']==2]['Voltage (V)'][::-1], cathode_am2_curve.loc[cathode_am2_curve['Step_ID']==2]['Absolute Capacity (mAh)'][::-1] * cathode_am2_irrev_scale * cathode_am2_rev_scale)
    interpCatCap22 = pd.DataFrame({'Capacity':x_interpCatCap22, 'Voltage':y_interpCatCap22})
    
    #Cathode AM3 Capacity
    cathode_am3_curve = pd.read_csv(os.path.join(half_cell_dir, 'Cathode_{0}.csv'.format(cathode_am3)))
    cathode_am3_curve['Absolute Capacity (mAh)'] = cathode_am3_curve['Specific Capacity (mAh/g)']*c_am3_grams
    #First charge -- Step 1
    y_interpCatCap31 = np.arange(0.5,5.0,0.001)
    x_interpCatCap31 = np.interp(y_interpCatCap31, cathode_am3_curve.loc[cathode_am3_curve['Step_ID']==1]['Voltage (V)'], cathode_am3_curve.loc[cathode_am3_curve['Step_ID']==1]['Absolute Capacity (mAh)'] * cathode_am3_irrev_scale)
    interpCatCap31 = pd.DataFrame({'Capacity':x_interpCatCap31, 'Voltage':y_interpCatCap31})
    #First discharge -- Step 2
    y_interpCatCap32 = np.arange(0.5,5.0,0.001)
    x_interpCatCap32 = np.interp(y_interpCatCap32, cathode_am3_curve.loc[cathode_am3_curve['Step_ID']==2]['Voltage (V)'][::-1], cathode_am3_curve.loc[cathode_am3_curve['Step_ID']==2]['Absolute Capacity (mAh)'][::-1] * cathode_am3_irrev_scale * cathode_am3_rev_scale)
    interpCatCap32 = pd.DataFrame({'Capacity':x_interpCatCap32, 'Voltage':y_interpCatCap32})

        
    # Cathode Curve
    cathodeX1Dfs = [interpCatCap11, interpCatCap21, interpCatCap31]
    cathodeX2Dfs = [interpCatCap12[::-1], interpCatCap22[::-1], interpCatCap32[::-1]]
    
    mergedCathodeX1Df = reduce(lambda left,right: pd.merge(left,right,on='Voltage'), cathodeX1Dfs)
    mergedCathodeX2Df = reduce(lambda left,right: pd.merge(left,right,on='Voltage'), cathodeX2Dfs)
    
    mergedCathodeX1Df['Combined'] = (mergedCathodeX1Df.Capacity_x + mergedCathodeX1Df.Capacity_y + mergedCathodeX1Df.Capacity)
    mergedCathodeX2Df['Combined'] = (mergedCathodeX2Df.Capacity_x + mergedCathodeX2Df.Capacity_y + mergedCathodeX2Df.Capacity)
    
    cathodeX1_lastValue = mergedCathodeX1Df['Combined'].iloc[-1]
    reversible_cathode_cap = mergedCathodeX2Df['Combined'].iloc[-1]
    cathode_areal_cap = reversible_cathode_cap / active_geo_area
    c_areal_cap.value = str(round(cathode_areal_cap, 2))
    
    mergedCathodeX2Df['Combined'] = mergedCathodeX2Df['Combined'] * -1 + cathodeX1_lastValue
#     cathodeElectrode = mergedCathodeX1Df.append(mergedCathodeX2Df)
    cathodeElectrode = pd.concat([mergedCathodeX1Df, mergedCathodeX2Df])
    
    cathodeElectrodeCapacity = cathodeElectrode['Combined'].tolist()
    cathodeElectrodeVoltage = cathodeElectrode['Voltage'].tolist()
    
    host.plot(cathodeElectrodeCapacity, cathodeElectrodeVoltage, 'b-', lw=2, label = 'Cathode Voltage')
    
    ############## Plotting Theoretical Anode Half-Cell Curve ##############    
    a_area = c_area * (1+overhang/100)
        
    a_am1_grams = (a_am1/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_am2_grams = (a_am2/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_am3_grams = (a_am3/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    
    a_addcomp1_grams = (a_ac1/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_addcomp2_grams = (a_ac2/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_condaid1_grams = (a_ca1/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_condaid2_grams = (a_ca2/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_binder1_grams = (a_bd1/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    a_binder2_grams = (a_bd2/100) * anode_ML * a_area * 2 * cathode_stacks / 1000
    
    total_a_am1_cost = a_am1_grams/1000 * a_am1_cost # in $
    total_a_am2_cost = a_am2_grams/1000 * a_am2_cost # in $
    total_a_am3_cost = a_am3_grams/1000 * a_am3_cost # in $
    total_a_addcomp1_cost = a_addcomp1_grams/1000 * a_addcomp1_cost # in $
    total_a_addcomp2_cost = a_addcomp2_grams/1000 * a_addcomp2_cost # in $
    total_a_condaid1_cost = a_condaid1_grams/1000 * a_condaid1_cost # in $
    total_a_condaid2_cost = a_condaid2_grams/1000 * a_condaid2_cost # in $
    total_a_binder1_cost = a_binder1_grams/1000 * a_binder1_cost # in $
    total_a_binder2_cost = a_binder2_grams/1000 * a_binder2_cost # in $


    #Anode AM1 Capacity
    anode_am1_curve = pd.read_csv(os.path.join(half_cell_dir, 'Anode_{0}.csv'.format(anode_am1)))
    anode_am1_curve['Absolute Capacity (mAh)'] = anode_am1_curve['Specific Capacity (mAh/g)']*a_am1_grams
    #First charge -- Step 1
    y_interpAndCap11 = np.arange(0,2.0,0.001)
    x_interpAndCap11 = np.interp(y_interpAndCap11, anode_am1_curve.loc[anode_am1_curve['Step_ID']==1]['Voltage (V)'][::-1], anode_am1_curve.loc[anode_am1_curve['Step_ID']==1]['Absolute Capacity (mAh)'][::-1] * anode_am1_irrev_scale)
    interpAndCap11 = pd.DataFrame({'Capacity':x_interpAndCap11, 'Voltage':y_interpAndCap11})
    #First discharge -- Step 2
    y_interpAndCap12 = np.arange(0,2.0,0.001)
    x_interpAndCap12 = np.interp(y_interpAndCap12, anode_am1_curve.loc[anode_am1_curve['Step_ID']==2]['Voltage (V)'], anode_am1_curve.loc[anode_am1_curve['Step_ID']==2]['Absolute Capacity (mAh)'] * anode_am1_irrev_scale * anode_am1_rev_scale)
    interpAndCap12 = pd.DataFrame({'Capacity':x_interpAndCap12, 'Voltage':y_interpAndCap12})    

    #Anode AM2 Capacity
    anode_am2_curve = pd.read_csv(os.path.join(half_cell_dir, 'Anode_{0}.csv'.format(anode_am2)))
    anode_am2_curve['Absolute Capacity (mAh)'] = anode_am2_curve['Specific Capacity (mAh/g)']*a_am2_grams
    #First charge -- Step 1
    y_interpAndCap21 = np.arange(0,2.0,0.001)
    x_interpAndCap21 = np.interp(y_interpAndCap21, anode_am2_curve.loc[anode_am2_curve['Step_ID']==1]['Voltage (V)'][::-1], anode_am2_curve.loc[anode_am2_curve['Step_ID']==1]['Absolute Capacity (mAh)'][::-1] * anode_am2_irrev_scale)
    interpAndCap21 = pd.DataFrame({'Capacity':x_interpAndCap21, 'Voltage':y_interpAndCap21})
    #First discharge -- Step 2
    y_interpAndCap22 = np.arange(0,2.0,0.001)
    x_interpAndCap22 = np.interp(y_interpAndCap22, anode_am2_curve.loc[anode_am2_curve['Step_ID']==2]['Voltage (V)'], anode_am2_curve.loc[anode_am2_curve['Step_ID']==2]['Absolute Capacity (mAh)'] * anode_am2_irrev_scale * anode_am2_rev_scale)
    interpAndCap22 = pd.DataFrame({'Capacity':x_interpAndCap22, 'Voltage':y_interpAndCap22})    
  
    
    #Anode AM3 Capacity
    anode_am3_curve = pd.read_csv(os.path.join(half_cell_dir, 'Anode_{0}.csv'.format(anode_am3)))
    anode_am3_curve['Absolute Capacity (mAh)'] = anode_am3_curve['Specific Capacity (mAh/g)']*a_am3_grams
    #First charge -- Step 1
    y_interpAndCap31 = np.arange(0,2.0,0.001)
    x_interpAndCap31 = np.interp(y_interpAndCap31, anode_am3_curve.loc[anode_am3_curve['Step_ID']==1]['Voltage (V)'][::-1], anode_am3_curve.loc[anode_am3_curve['Step_ID']==1]['Absolute Capacity (mAh)'][::-1] * anode_am3_irrev_scale)
    interpAndCap31 = pd.DataFrame({'Capacity':x_interpAndCap31, 'Voltage':y_interpAndCap31})
    #First discharge -- Step 2
    y_interpAndCap32 = np.arange(0,2.0,0.001)
    x_interpAndCap32 = np.interp(y_interpAndCap32, anode_am3_curve.loc[anode_am3_curve['Step_ID']==2]['Voltage (V)'], anode_am3_curve.loc[anode_am3_curve['Step_ID']==2]['Absolute Capacity (mAh)'] * anode_am3_irrev_scale * anode_am3_rev_scale)
    interpAndCap32 = pd.DataFrame({'Capacity':x_interpAndCap32, 'Voltage':y_interpAndCap32})          
    
    # Anode Curve
    anodeX1Dfs = [interpAndCap11[::-1], interpAndCap21[::-1], interpAndCap31[::-1]]
    anodeX2Dfs = [interpAndCap12, interpAndCap22, interpAndCap32]
    
    mergedAnodeX1Df = reduce(lambda left,right: pd.merge(left,right,on='Voltage'), anodeX1Dfs)
    mergedAnodeX2Df = reduce(lambda left,right: pd.merge(left,right,on='Voltage'), anodeX2Dfs)
    
    mergedAnodeX1Df['Combined'] = (mergedAnodeX1Df.Capacity_x + mergedAnodeX1Df.Capacity_y + mergedAnodeX1Df.Capacity)
    mergedAnodeX2Df['Combined'] = (mergedAnodeX2Df.Capacity_x + mergedAnodeX2Df.Capacity_y + mergedAnodeX2Df.Capacity)
    
    anodeX1_lastValue = mergedAnodeX1Df['Combined'].iloc[-1]
    mergedAnodeX2Df['Combined'] = mergedAnodeX2Df['Combined'] * -1 + anodeX1_lastValue
#     anodeElectrode = mergedAnodeX1Df.append(mergedAnodeX2Df)
    anodeElectrode = pd.concat([mergedAnodeX1Df, mergedAnodeX2Df])
    
    anodeElectrodeCapacity = anodeElectrode['Combined'].tolist()
    anodeElectrodeVoltage = anodeElectrode['Voltage'].tolist()
    
    
    total_cell_cost = sum([total_a_am1_cost, total_a_am2_cost, total_a_am3_cost,
                           total_a_addcomp1_cost, total_a_addcomp2_cost,
                           total_a_condaid1_cost, total_a_condaid2_cost,
                           total_a_binder1_cost, total_a_binder2_cost,
                           total_c_am1_cost, total_c_am2_cost, total_c_am3_cost,
                           total_c_addcomp1_cost, total_c_addcomp2_cost,
                           total_c_condaid1_cost, total_c_condaid2_cost,
                           total_c_binder1_cost, total_c_binder2_cost,
                           total_anode_cc_cost, total_cathode_cc_cost,
                           total_separator_cost, total_elyte_cost, total_pouch_cost,
                           total_pos_term_cost, total_neg_term_cost
                          ])
    
    
    host.plot(anodeElectrodeCapacity, anodeElectrodeVoltage, 'r-', lw=2, label = 'Anode Voltage')

    
    ######## Theoretical Full Cell Curve ###########
    finalAndNewx = np.arange(0,start_capacity + target_capacity)
    finalAndNewy1 = np.interp(finalAndNewx, mergedAnodeX1Df['Combined'], mergedAnodeX1Df['Voltage'])
    interpFinalAnd1 = pd.DataFrame({'Capacity':finalAndNewx, 'Voltage':finalAndNewy1})
    
    finalCatNewx = np.arange(0,start_capacity + target_capacity)
    finalCatNewy1 = np.interp(finalCatNewx, mergedCathodeX1Df['Combined'], mergedCathodeX1Df['Voltage'])
    interpFinalCat1 = pd.DataFrame({'Capacity':finalCatNewx, 'Voltage':finalCatNewy1})
        
    electrodesDf1 = [interpFinalCat1, interpFinalAnd1]
    
    finalAndNewx = np.arange(start_capacity,start_capacity + target_capacity)
    finalAndNewy2 = np.interp(finalAndNewx, mergedAnodeX2Df['Combined'].iloc[::-1], mergedAnodeX2Df['Voltage'].iloc[::-1])
    interpFinalAnd2 = pd.DataFrame({'Capacity':finalAndNewx, 'Voltage':finalAndNewy2})
    
    finalCatNewx = np.arange(start_capacity,start_capacity + target_capacity)
    finalCatNewy2 = np.interp(finalCatNewx, mergedCathodeX2Df['Combined'].iloc[::-1], mergedCathodeX2Df['Voltage'].iloc[::-1])
    interpFinalCat2 = pd.DataFrame({'Capacity':finalCatNewx, 'Voltage':finalCatNewy2})
        
    electrodesDf2 = [interpFinalCat2, interpFinalAnd2]
    
    fullCellDf1 = reduce(lambda left,right: pd.merge(left,right,on='Capacity'), electrodesDf1)
    fullCellDf2 = reduce(lambda left,right: pd.merge(left,right,on='Capacity'), electrodesDf2)

    fullCellDf1["Differenced"] =  fullCellDf1.Voltage_x - fullCellDf1.Voltage_y 
    fullCellDf2["Differenced"] =  fullCellDf2.Voltage_x - fullCellDf2.Voltage_y 
    
    cellCapacity1 = fullCellDf1['Capacity'].tolist()
    cellVoltage1 = fullCellDf1['Differenced'].tolist()
    
    cellCapacity2 = fullCellDf2['Capacity'].tolist()
    cellVoltage2 = fullCellDf2['Differenced'].tolist()
    
    host.plot(cellCapacity1, cellVoltage1,'k-', lw=4, label = 'Theoretical Full Cell Voltage') 
    host.plot(cellCapacity2, cellVoltage2,'k-', lw=4) 

    max_capacity = max(cellCapacity2)
    socVals = [x/max_capacity for x in cellCapacity2]

    f = interpolate.interp1d(socVals, cellVoltage2, fill_value = "extrapolate")
    
    soc_0.value = str(u_min)
    soc_10.value = str(f(0.1))
    soc_20.value = str(f(0.2))
    soc_30.value = str(f(0.3))
    soc_40.value = str(f(0.4))
    soc_50.value = str(f(0.5))
    soc_60.value = str(f(0.6))
    soc_70.value = str(f(0.7))
    soc_80.value = str(f(0.8))
    soc_90.value = str(f(0.9))
    soc_100.value = str(u_max)
    
    plt.title('Electrode Balancing & Theoretical Full Cell Voltage Derivation', fontsize=16)
    plt.xlabel('Capacity [mAh]', fontsize=15)
    plt.ylabel('Voltage [V]', fontsize=15)
    if u_max <= 4.5:
        upper_limit = 4.5
    else:
        upper_limit = u_max + 0.2
    host.axis((-20,target_capacity*2+start_capacity,0,upper_limit))
#     host.axis((-20,150000,0,upper_limit))
        
    #drawing Cell Balance line
    host.axhline(y=u_max, color='b', linewidth=0.5, linestyle='--')
    #drawing Min Voltage line
    host.axhline(y=u_min,color='b', linewidth=0.5, linestyle='--')
    #drawing Start Capacity Line
    host.axvline(x=start_capacity, color='b', linewidth=1, linestyle='-')
    #drawing Target Capacity Line
    host.axvline(x=start_capacity+target_capacity, color='b', linewidth=1, linestyle='-')
    host.axvline(x=(start_capacity+target_capacity)*n_p_ratio, color='b', linewidth=0.5, linestyle='--')
    host.grid(which='major', ls=':', lw=0.1, color='gray')

    
#     ############## Plotting Anode Subplot ##############
#     ax2 = plt.axes([1,0.15,0.5,0.5])
#     ax2.plot(anodeElectrodeCapacity, anodeElectrodeVoltage, 'r-', lw=2)
#     plt.setp(ax2, xticks = [target_capacity+start_capacity], yticks=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
#     ax2.set_xticklabels([target_capacity])
#     plt.axis((0,target_capacity*2+start_capacity,0,0.7))
#     plt.title('Anode Detail', y=1.15)
#     plt.ylabel('V')
#     plt.xlabel('mAh')
#     ax2.axvline(x=start_capacity, ls='-', color='b', lw=1)
#     ax2.axvline(x=start_capacity+target_capacity, ls='-', color='b', lw=1)
#     ax2.axvline(x=(start_capacity+target_capacity)*n_p_ratio, ls='--', color='b', lw=1)

#     ax3 = ax2.twiny()
#     new_tick_locations = np.array([target_capacity*0.05+start_capacity, target_capacity*0.2+start_capacity, target_capacity*0.5+start_capacity, target_capacity*0.8+start_capacity, target_capacity+start_capacity])
#     ax3.set_xlim(ax2.get_xlim())
#     ax3.set_xticks(new_tick_locations)
#     ax3.set_xticklabels(['5%', '20%', '50%', '80%', '100%'])
#     ax3.set_xlabel("State-of-Charge")    
    
    
    ############## Plotting Cathode Subplot ##############
#     ax4 = plt.axes([1.6,0.15,0.5,0.5])
    ax4 = plt.axes([1,0.15,0.5,0.5])
    ax4.plot(cellCapacity1, cellVoltage1,'k-', lw=4)
    ax4.plot(cellCapacity2, cellVoltage2,'k-', lw=4)
    plt.axis((0,target_capacity*1.5+start_capacity,u_max-0.4,u_max+0.2))
    plt.title('Top-of-Charge Detail', y=1.15)
    plt.ylabel('V')
    plt.xlabel('mAh')
    ax4.axvline(x=start_capacity+target_capacity, ls='-', color='b', lw=1)
    ax4.axhline(y=u_max, ls='-', color='b', lw=1)
    
    
    ############### Matching Curves ######################
    if matching_curve_file:
        matching_curve_df = pd.read_csv(matching_curve_file)
        matching_curve_df['capacity'] = matching_curve_df['capacity'].apply(lambda x: x * matching_curve_grams)
        matching_curve_df['capacity'] = matching_curve_df['capacity']
        matching_curve_cap = matching_curve_df['capacity'].tolist()
        matching_curve_volt = matching_curve_df['voltage'].tolist()
        host.plot(matching_curve_cap, matching_curve_volt, ls='-', lw='3', color='orange', label = 'Experimental Full Cell Voltage')
        ax4.plot(matching_curve_cap, matching_curve_volt, ls='-', lw='3', color='orange')

    watt_hours = trapz(cellVoltage2, cellCapacity2)/1000
    cell_energy.value = str(round(watt_hours,2))
    specific_energy.value = str(round(watt_hours/(total_cell_mass/1000),2))
    energy_density.value = str(round(watt_hours/total_cell_volume, 2))
    cell_cost.value = str(round(total_cell_cost,2))
    cell_normalized_cost.value = str(round((total_cell_cost/(watt_hours/1000)),1))
    
    ############ Plotting Cost Breakdown Pie Chart ############
    ax5 = plt.axes([1.4,0.05,0.8,0.8])
    pie_labels = ['CAM1', 'CAM2', 'CAM3', 'AAM1', 'AAM2', 'AAM3', '+CBD', '-CBD', 'SEP', 'ELY', '+CC', '-CC', 'OTH']
    pie_costs = [total_c_am1_cost, total_c_am2_cost, total_c_am3_cost,
                 total_a_am1_cost, total_a_am2_cost, total_a_am3_cost,
                 sum([total_c_addcomp1_cost, total_c_addcomp2_cost, total_c_condaid1_cost, total_c_condaid2_cost, total_c_binder1_cost, total_c_binder2_cost]),
                 sum([total_a_addcomp1_cost, total_a_addcomp2_cost, total_a_condaid1_cost, total_a_condaid2_cost, total_a_binder1_cost, total_a_binder2_cost]),
                 total_separator_cost,
                 total_elyte_cost, 
                 total_cathode_cc_cost, total_anode_cc_cost,
                 sum([total_pouch_cost, total_pos_term_cost, total_neg_term_cost])
                ]
    pie_labels_cleaned = []
    pie_costs_cleaned = []
    i=0
    for component in pie_costs:
        if component > 0:
            pie_costs_cleaned.append(component)
            pie_labels_cleaned.append(pie_labels[i])
        i+=1
    
    ax5.set_title('Cell Cost Breakdown\nCost = ${}/kWh'.format(round((total_cell_cost/(watt_hours/1000)),1)))
    explode = [0.015]*len(pie_costs_cleaned)
    if sum(pie_costs) > 0:
        ax5.pie(pie_costs_cleaned, labels=pie_labels_cleaned, autopct='%1.1f%%', explode=explode, normalize=True)
    
        
    full_voltage_vals1.value = str(cellVoltage1)
    full_capacity_vals1.value = str(cellCapacity1)
    full_voltage_vals2.value = str(cellVoltage2)
    full_capacity_vals2.value = str(cellCapacity2)
    negative_voltage_vals.value = str(anodeElectrodeVoltage)
    negative_capacity_vals.value = str(anodeElectrodeCapacity)
    positive_voltage_vals.value = str(cathodeElectrodeVoltage)
    positive_capacity_vals.value = str(cathodeElectrodeCapacity)
    
    host.legend(loc=9, bbox_to_anchor=(0.5, -0.1), ncol=2)
    
figure = wg.interactive_output(plot_main_fig, 
            {'u_max': u_max,
            'u_min': u_min,
            'n_p_ratio' : n_p_ratio,
            'target_capacity': target_capacity, 
            'start_capacity': start_capacity,
            'anode_am1': anode_am1,
            'anode_am2': anode_am2, 
            'anode_am3': anode_am3,
            'anode_am1_irrev_scale': anode_am1_irrev_scale,
            'anode_am1_rev_scale': anode_am1_rev_scale,             
            'anode_am2_irrev_scale': anode_am2_irrev_scale,
            'anode_am2_rev_scale': anode_am2_rev_scale,             
            'anode_am3_irrev_scale': anode_am3_irrev_scale,
            'anode_am3_rev_scale': anode_am3_rev_scale,             
            'a_am1': a_am1_percentage, 
            'a_am2': a_am2_percentage, 
            'a_am3': a_am3_percentage, 
            'a_ac1': a_addcomp1_percentage, 
            'a_ac2': a_addcomp2_percentage, 
            'a_ca1': a_condaid1_percentage, 
            'a_ca2': a_condaid2_percentage,
            'a_bd1': a_binder1_percentage, 
            'a_bd2': a_binder2_percentage, 
            'a_am1_density': a_am1_density,
            'a_am2_density': a_am2_density,
            'a_am3_density': a_am3_density,
            'a_addcomp1_density': a_addcomp1_density,
            'a_addcomp2_density': a_addcomp2_density,
            'a_condaid1_density': a_condaid1_density,
            'a_condaid2_density': a_condaid2_density,
            'a_binder1_density': a_binder1_density,
            'a_binder2_density': a_binder2_density,
            'a_density': a_density,
            'cathode_am1': cathode_am1, 
            'cathode_am2': cathode_am2, 
            'cathode_am3': cathode_am3,
            'cathode_am1_irrev_scale': cathode_am1_irrev_scale,
            'cathode_am1_rev_scale': cathode_am1_rev_scale,
            'cathode_am2_irrev_scale': cathode_am2_irrev_scale,
            'cathode_am2_rev_scale': cathode_am2_rev_scale,
            'cathode_am3_irrev_scale': cathode_am3_irrev_scale,
            'cathode_am3_rev_scale': cathode_am3_rev_scale,
            'c_am1': c_am1_percentage, 
            'c_am2': c_am2_percentage, 
            'c_am3': c_am3_percentage, 
            'c_ac1': c_addcomp1_percentage, 
            'c_ac2': c_addcomp2_percentage, 
            'c_ca1': c_condaid1_percentage, 
            'c_ca2': c_condaid2_percentage,
            'c_bd1': c_binder1_percentage, 
            'c_bd2': c_binder2_percentage, 
            'c_am1_density': c_am1_density,
            'c_am2_density': c_am2_density,
            'c_am3_density': c_am3_density,
            'c_addcomp1_density': c_addcomp1_density,
            'c_addcomp2_density': c_addcomp2_density,
            'c_condaid1_density': c_condaid1_density,
            'c_condaid2_density': c_condaid2_density,
            'c_binder1_density': c_binder1_density,
            'c_binder2_density': c_binder2_density,
            'c_density': c_density,
            'anode_ML': anode_ML_slider, 
            'cathode_ML': cathode_ML_slider, 
            'cathode_stacks': cathode_stacks_slider, 
            'c_area': c_area, 
            'overhang': overhang,
            'matching_curve_file': matching_curve_file,
            'matching_curve_grams': matching_curve_grams,
            'cathode_uncoated_area': cathode_uncoated_area,
            'cathode_cc_thickness': cathode_cc_thickness, 
            'cathode_cc_density': cathode_cc_density,
            'anode_coated_area': anode_coated_area, 
            'anode_uncoated_area': anode_uncoated_area, 
            'anode_cc_thickness': anode_cc_thickness, 
            'anode_cc_density': anode_cc_density,
            'separator_thickness': separator_thickness, 
            'separator_width': separator_width, 
            'separator_fold_length': separator_fold_length, 
            'separator_density': separator_density, 
            'separator_porosity': separator_porosity,
            'elyte_overhead': elyte_overhead, 
            'elyte_density': elyte_density, 
            'c_swell_factor': c_swell_factor, 
            'a_swell_factor': a_swell_factor,
            'mylar_thickness': mylar_thickness, 
            'mylar_areal_mass': mylar_areal_mass, 
            'pos_terminal_mass': pos_terminal_mass, 
            'neg_terminal_mass': neg_terminal_mass, 
            'other_mass': other_mass, 
            'seal_1_length': seal_1_length, 
            'seal_2_length': seal_2_length, 
            'seal_3_length': seal_3_length, 
            'seal_4_length': seal_4_length, 
            'pouch_length': pouch_length, 
            'pouch_width': pouch_width,
            'a_am1_cost': a_am1_cost, 
            'a_am2_cost': a_am2_cost, 
            'a_am3_cost': a_am3_cost, 
            'a_addcomp1_cost': a_addcomp1_cost, 
            'a_addcomp2_cost': a_addcomp2_cost, 
            'a_condaid1_cost': a_condaid1_cost, 
            'a_condaid2_cost': a_condaid2_cost,
            'a_binder1_cost': a_binder1_cost, 
            'a_binder2_cost': a_binder2_cost,
            'c_am1_cost': c_am1_cost, 
            'c_am2_cost': c_am2_cost, 
            'c_am3_cost': c_am3_cost, 
            'c_addcomp1_cost': c_addcomp1_cost, 
            'c_addcomp2_cost': c_addcomp2_cost, 
            'c_condaid1_cost': c_condaid1_cost, 
            'c_condaid2_cost': c_condaid2_cost,
            'c_binder1_cost': c_binder1_cost, 
            'c_binder2_cost': c_binder2_cost,
            'cathode_cc_cost': cathode_cc_cost, 
            'anode_cc_cost': anode_cc_cost, 
            'separator_cost': separator_cost, 
            'elyte_cost': elyte_cost, 
            'mylar_cost': mylar_cost, 
            'pos_terminal_cost': pos_terminal_cost, 
            'neg_terminal_cost': neg_terminal_cost
            })


def export(filename):
    if filename:
        unallowed_chars = ['/', ':', '*', '?', '"', '<', '>', '|']
        flag = 0
        for char in unallowed_chars:
            if filename.find(char) >= 0:
                flag += 1
        
        if flag > 0:
            print('Please make sure filename is valid')
        else:
            export_values_dict = {
                    'u_max': u_max.value,
                    'u_min': u_min.value,
                    'n_p_ratio' : n_p_ratio.value,
                    'target_capacity': target_capacity.value, 
                    'start_capacity': start_capacity.value,
                    'anode_am1': anode_am1.value,
                    'anode_am2': anode_am2.value, 
                    'anode_am3': anode_am3.value,
                    'anode_am1_irrev_scale': anode_am1_irrev_scale.value,
                    'anode_am1_rev_scale': anode_am1_rev_scale.value,             
                    'anode_am2_irrev_scale': anode_am2_irrev_scale.value,
                    'anode_am2_rev_scale': anode_am2_rev_scale.value,             
                    'anode_am3_irrev_scale': anode_am3_irrev_scale.value,
                    'anode_am3_rev_scale': anode_am3_rev_scale.value,             
                    'a_am1': a_am1_percentage.value, 
                    'a_am2': a_am2_percentage.value, 
                    'a_am3': a_am3_percentage.value, 
                    'a_ac1': a_addcomp1_percentage.value, 
                    'a_ac2': a_addcomp2_percentage.value, 
                    'a_ca1': a_condaid1_percentage.value, 
                    'a_ca2': a_condaid2_percentage.value,
                    'a_bd1': a_binder1_percentage.value, 
                    'a_bd2': a_binder2_percentage.value, 
                    'a_am1_density': a_am1_density.value,
                    'a_am2_density': a_am2_density.value,
                    'a_am3_density': a_am3_density.value,
                    'a_addcomp1_density': a_addcomp1_density.value,
                    'a_addcomp2_density': a_addcomp2_density.value,
                    'a_condaid1_density': a_condaid1_density.value,
                    'a_condaid2_density': a_condaid2_density.value,
                    'a_binder1_density': a_binder1_density.value,
                    'a_binder2_density': a_binder2_density.value,
                    'a_density': a_density.value,
                    'cathode_am1': cathode_am1.value, 
                    'cathode_am2': cathode_am2.value, 
                    'cathode_am3': cathode_am3.value,
                    'cathode_am1_irrev_scale': cathode_am1_irrev_scale.value,
                    'cathode_am1_rev_scale': cathode_am1_rev_scale.value,
                    'cathode_am2_irrev_scale': cathode_am2_irrev_scale.value,
                    'cathode_am2_rev_scale': cathode_am2_rev_scale.value,
                    'cathode_am3_irrev_scale': cathode_am3_irrev_scale.value,
                    'cathode_am3_rev_scale': cathode_am3_rev_scale.value,
                    'c_am1': c_am1_percentage.value, 
                    'c_am2': c_am2_percentage.value, 
                    'c_am3': c_am3_percentage.value, 
                    'c_ac1': c_addcomp1_percentage.value, 
                    'c_ac2': c_addcomp2_percentage.value, 
                    'c_ca1': c_condaid1_percentage.value, 
                    'c_ca2': c_condaid2_percentage.value,
                    'c_bd1': c_binder1_percentage.value, 
                    'c_bd2': c_binder2_percentage.value, 
                    'c_am1_density': c_am1_density.value,
                    'c_am2_density': c_am2_density.value,
                    'c_am3_density': c_am3_density.value,
                    'c_addcomp1_density': c_addcomp1_density.value,
                    'c_addcomp2_density': c_addcomp2_density.value,
                    'c_condaid1_density': c_condaid1_density.value,
                    'c_condaid2_density': c_condaid2_density.value,
                    'c_binder1_density': c_binder1_density.value,
                    'c_binder2_density': c_binder2_density.value,
                    'c_density': c_density.value,
                    'anode_ML': anode_ML_slider.value, 
                    'cathode_ML': cathode_ML_slider.value, 
                    'cathode_stacks': cathode_stacks_slider.value, 
                    'c_area': c_area.value, 
                    'overhang': overhang.value,
                    'cathode_uncoated_area': cathode_uncoated_area.value,
                    'cathode_cc_thickness': cathode_cc_thickness.value, 
                    'cathode_cc_density': cathode_cc_density.value,
                    'anode_coated_area': anode_coated_area.value, 
                    'anode_uncoated_area': anode_uncoated_area.value, 
                    'anode_cc_thickness': anode_cc_thickness.value, 
                    'anode_cc_density': anode_cc_density.value,
                    'separator_thickness': separator_thickness.value, 
                    'separator_width': separator_width.value, 
                    'separator_fold_length': separator_fold_length.value, 
                    'separator_density': separator_density.value, 
                    'separator_porosity': separator_porosity.value,
                    'elyte_overhead': elyte_overhead.value, 
                    'elyte_density': elyte_density.value, 
                    'c_swell_factor': c_swell_factor.value, 
                    'a_swell_factor': a_swell_factor.value,
                    'mylar_thickness': mylar_thickness.value, 
                    'mylar_areal_mass': mylar_areal_mass.value, 
                    'pos_terminal_mass': pos_terminal_mass.value, 
                    'neg_terminal_mass': neg_terminal_mass.value, 
                    'other_mass': other_mass.value, 
                    'seal_1_length': seal_1_length.value, 
                    'seal_2_length': seal_2_length.value, 
                    'seal_3_length': seal_3_length.value, 
                    'seal_4_length': seal_4_length.value, 
                    'pouch_length': pouch_length.value, 
                    'pouch_width': pouch_width.value,
                    'soc_0': soc_0.value,
                    'soc_10': soc_10.value,
                    'soc_20': soc_20.value,
                    'soc_30': soc_30.value,
                    'soc_40': soc_40.value,
                    'soc_50': soc_50.value,
                    'soc_60': soc_60.value,
                    'soc_70': soc_70.value,
                    'soc_80': soc_80.value,
                    'soc_90': soc_90.value,
                    'soc_100': soc_100.value,
            }
            
            with open('{0}.csv'.format(filename), 'w') as f:
                writer = csv.writer(f)
                writer.writerow(["Parameter", "Value"])
                for key in export_values_dict.keys():
                    f.write("%s,%s\n"%(key,export_values_dict[key]))            
    else:
        print('Please enter a .csv filename prior to exporting')

def export_voltage_curves(filename):
    if filename:
        unallowed_chars = ['/', ':', '*', '?', '"', '<', '>', '|']
        flag = 0
        for char in unallowed_chars:
            if filename.find(char) >= 0:
                flag += 1
        if flag > 0:
            print('Please make sure filename is valid')
        else:
            voltage_curve_dict = {'Full Cell Charge Voltage': eval(full_voltage_vals1.value),
                                  'Full Cell Charge Capacity': eval(full_capacity_vals1.value),
                                  'Full Cell Discharge Voltage': eval(full_voltage_vals2.value),
                                  'Full Cell Discharge Capacity': eval(full_capacity_vals2.value),
                                  'Positive Voltage': eval(positive_voltage_vals.value),
                                  'Positive Capacity': eval(positive_capacity_vals.value),
                                  'Negative Voltage': eval(negative_voltage_vals.value),
                                  'Negative Capacity': eval(negative_capacity_vals.value),
                                 }
            export_voltage_df = pd.DataFrame(dict([ (k,pd.Series(v)) for k,v in voltage_curve_dict.items() ]))
            export_voltage_df.to_csv('{}_voltage_curves.csv'.format(filename))            
    else:
        print('Please enter a .csv filename prior to exporting')

def import_params(filepath):
    if filepath:
        try:
            load_cell_params = pd.read_csv(filepath)
            params_list = load_cell_params['Value'].tolist()
            u_max.value = params_list[0]
            u_min.value = params_list[1] 
            n_p_ratio.value = params_list[2]
            target_capacity.value = params_list[3]
            start_capacity.value = params_list[4]
            anode_am1.value = params_list[5]
            anode_am2.value = params_list[6]
            anode_am3.value = params_list[7]
            anode_am1_irrev_scale.value = params_list[8]
            anode_am1_rev_scale.value = params_list[9]
            anode_am2_irrev_scale.value = params_list[10]
            anode_am2_rev_scale.value = params_list[11]
            anode_am3_irrev_scale.value = params_list[12]
            anode_am3_rev_scale.value = params_list[13]
            a_am1_percentage.value = params_list[14]
            a_am2_percentage.value = params_list[15]
            a_am3_percentage.value = params_list[16]
            a_addcomp1_percentage.value = params_list[17]
            a_addcomp2_percentage.value = params_list[18]
            a_condaid1_percentage.value = params_list[19]
            a_condaid2_percentage.value = params_list[20]
            a_binder1_percentage.value = params_list[21]
            a_binder2_percentage.value = params_list[22]
            a_am1_density.value = params_list[23]
            a_am2_density.value = params_list[24]
            a_am3_density.value = params_list[25]
            a_addcomp1_density.value = params_list[26]
            a_addcomp2_density.value = params_list[27]
            a_condaid1_density.value = params_list[28]
            a_condaid2_density.value = params_list[29]
            a_binder1_density.value = params_list[30]
            a_binder2_density.value = params_list[31]
            a_density.value = params_list[32]
            cathode_am1.value = params_list[33]
            cathode_am2.value = params_list[34]
            cathode_am3.value = params_list[35]
            cathode_am1_irrev_scale.value = params_list[36]
            cathode_am1_rev_scale.value = params_list[37]
            cathode_am2_irrev_scale.value = params_list[38]
            cathode_am2_rev_scale.value = params_list[39]
            cathode_am3_irrev_scale.value = params_list[40]
            cathode_am3_rev_scale.value = params_list[41]
            c_am1_percentage.value = params_list[42]
            c_am2_percentage.value = params_list[43]
            c_am3_percentage.value = params_list[44]
            c_addcomp1_percentage.value = params_list[45]
            c_addcomp2_percentage.value = params_list[46]
            c_condaid1_percentage.value = params_list[47]
            c_condaid2_percentage.value = params_list[48]
            c_binder1_percentage.value = params_list[49]
            c_binder2_percentage.value = params_list[50]
            c_am1_density.value = params_list[51]
            c_am2_density.value = params_list[52]
            c_am3_density.value = params_list[53]
            c_addcomp1_density.value = params_list[54]
            c_addcomp2_density.value = params_list[55]
            c_condaid1_density.value = params_list[56]
            c_condaid2_density.value = params_list[57]
            c_binder1_density.value = params_list[58]
            c_binder2_density.value = params_list[59]
            c_density.value = params_list[60]
            anode_ML_slider.value = params_list[61]
            cathode_ML_slider.value = params_list[62]
            cathode_stacks_slider.value = params_list[63]
            c_area.value = params_list[64]
            overhang.value = params_list[65]
            cathode_uncoated_area.value = params_list[66]
            cathode_cc_thickness.value = params_list[67]
            cathode_cc_density.value = params_list[68]
            anode_coated_area.value = params_list[69]
            anode_uncoated_area.value = params_list[70]
            anode_cc_thickness.value = params_list[71]
            anode_cc_density.value = params_list[72]
            separator_thickness.value = params_list[73]
            separator_width.value = params_list[74]
            separator_fold_length.value = params_list[75]
            separator_density.value = params_list[76]
            separator_porosity.value = params_list[77]
            elyte_overhead.value = params_list[78]
            elyte_density.value = params_list[79]
            c_swell_factor.value = params_list[80]
            a_swell_factor.value = params_list[81]
            mylar_thickness.value = params_list[82]
            mylar_areal_mass.value = params_list[83]
            pos_terminal_mass.value = params_list[84]
            neg_terminal_mass.value = params_list[85]
            other_mass.value = params_list[86]
            seal_1_length.value = params_list[87]
            seal_2_length.value = params_list[88]
            seal_3_length.value = params_list[89]
            seal_4_length.value = params_list[90]
            pouch_length.value = params_list[91]
            pouch_width.value = params_list[92]
        except:
            print('An error occured. Please make sure cell design parameters are properly configured.')

    else:
        print('Please enter a valid filepath to load cell design parameters')
    
def import_matching_curve(filepath):
    if filepath:
        matching_curve_file.value = filepath
    else:
        matching_curve_file.value = ''

wg.interact_manual.opts['manual_name'] = 'Import Cell Design'
import_button = wg.interact_manual(import_params,filepath=Text(placeholder='.csv filename path', description='File Path:', layout=wg.Layout(width='50%')))

wg.interact_manual.opts['manual_name'] = 'Match Curve'
import_matching_button = wg.interact_manual(import_matching_curve, filepath=Text(placeholder='.csv filename path', description='File Path:', layout=wg.Layout(width='50%')))

tab_titles = [
    'Cell Level Details',
    'Cathode Formulation',
    'Anode Formulation',
    'Electrode Details',
    'Other Design Inputs',
    'Final Cell Output',
    'Matching Curve'
]
tab_children = [
    wg.VBox([cell_balance_box, target_capacity_box, start_capacity_box], margin='10px 0px 0px 10px'),
    wg.VBox([wg.HBox([cathodeFormulationDetails, cathode_materials])]),
    wg.VBox([wg.HBox([anodeFormulationDetails, anode_materials])]),
    wg.VBox([cathode_ML_slider_box, anode_ML_slider_box, cathode_stacks_slider_box, c_area_box, overhang_box, c_ss_thickness_box, a_ss_thickness_box]),
    wg.VBox([cathode_details_box, anode_details_box, separator_details_box, elyte_swell_box, inactive_details_box]),
    wg.VBox([cell_mass_box, cell_thickness_box, cell_energy_box, cell_specific_energy_box, cell_energy_density_box, cell_cost_box, cell_normalized_cost_box]),
    wg.VBox([matching_curve_grams]),
]
params_tabs = wg.Tab(tab_children, titles = tab_titles)

final_display = wg.VBox([params_tabs, figure])
display(final_display)

wg.interact_manual.opts['manual_name'] = 'Export Cell Design'
export_button = wg.interact_manual(export,filename=Text(placeholder='cell_design_name (omit ".csv")', description='File Name:',  layout=wg.Layout(width='50%')))

wg.interact_manual.opts['manual_name'] = 'Export Voltage Curves'
export_button = wg.interact_manual(export_voltage_curves,filename=Text(placeholder='cell_design_name (omit ".csv")', description='File Name:',  layout=wg.Layout(width='50%')))